In [2]:
from google import genai
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

# This is the sentence we want to turn into numbers
sentence = "I love playing football."

# ask to convert the sentence into a vector
result = client.models.embed_content(
    model="gemini-embedding-001",  # the specific AI model that does this conversion
    contents=sentence
)

# Pulls the actual list of numbers out of the result
embedding = result.embeddings[0].values

# Print things out so we can see what happened
print("Sentence:", sentence)
print("First 5 numbers of its vector:", embedding[:5])  # just a peek, the full list is huge
print("Total numbers in the vector:", len(embedding))   # usually 768 or 3072 numbers

Sentence: I love playing football.
First 5 numbers of its vector: [-0.012424914, 0.0066163046, 0.021517763, -0.06809436, -0.01598877]
Total numbers in the vector: 3072


In [1]:
pip install google-genai python-dotenv scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from google import genai
from dotenv import load_dotenv
import os
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# A tiny helper function so we don't repeat the same 4 lines every time
def get_embedding(text):
    # Send text to the model, get back numbers
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    # Return just the list of numbers (not the whole response object)
    return result.embeddings[0].values

# Two sentences that MEAN similar things but use different words
sentence1 = "I love playing football."
sentence2 = "I enjoy playing soccer."

# Convert both to numbers using our helper function
vector1 = get_embedding(sentence1)
vector2 = get_embedding(sentence2)

# cosine_similarity expects lists-of-lists, that's why vector1 and vector2
# are wrapped in extra square brackets: [vector1], [vector2]
similarity = cosine_similarity([vector1], [vector2])

# The result comes back as a mini grid (matrix), so we dig into [0][0]
# to get the single number we actually care about
score = similarity[0][0]

print("Sentence 1:", sentence1)
print("Sentence 2:", sentence2)
print("How similar are they? (closer to 1 = more similar):", score)

Sentence 1: I love playing football.
Sentence 2: I enjoy playing soccer.
How similar are they? (closer to 1 = more similar): 0.8180875413935074


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
import time
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def get_embedding(text):
    result = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return result.embeddings[0].values

# Our little "database" of sentences
sentences = [
    "I love playing football.",
    "Python is a programming language.",
    "I enjoy playing soccer."
]

# We need one vector per sentence, so we loop through and collect them
embeddings = []
for sentence in sentences:
    vector = get_embedding(sentence)
    embeddings.append(vector)
    time.sleep(2)

# Now compare EVERY sentence to EVERY other sentence in one go
# This gives us a grid: matrix[i][j] = how similar sentence i is to sentence j
similarity_matrix = cosine_similarity(embeddings)

print("Sentences:")
for i, sentence in enumerate(sentences):
    print(i, "-", sentence)

print("\nSimilarity grid:")
print(similarity_matrix)
# Reading tip: matrix[0][2] tells you how similar sentence 0 is to sentence 2
# The diagonal (matrix[0][0], matrix[1][1]...) is always 1.0 -- a sentence is
# perfectly similar to itself

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Quota exceeded for aiplatform.googleapis.com/global_embed_content_requests_per_minute_per_base_model with base model: gemini-embedding. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.', 'status': 'RESOURCE_EXHAUSTED'}}